# Notebook 2: Create Labels
**MLOps Training 2026/2027 · Task 2: From Tables to Notebooks**

### Objective:
1. Load the joined master ML table from Notebook 1 (`joined_orders.parquet`).
2. Filter the population to orders with a definitive delivery outcome (`order_status == 'delivered'` with valid timestamps).
3. Define and construct the binary target label **`is_late`**:
   - `1` (Late): `order_delivered_customer_date > order_estimated_delivery_date`
   - `0` (On-time / Early): `order_delivered_customer_date <= order_estimated_delivery_date`
4. Compute delivery duration metrics (`actual_delivery_days`, `estimated_delivery_days`, `delivery_delay_days`) to analyze delivery dynamics.
5. Analyze the **class imbalance** and distribution of late deliveries across time and geographical regions.
6. Validate data integrity and export the labeled dataset artifact (`labeled_orders.parquet`) for Notebook 3 (`Split`).

In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from IPython.display import display

# Display configuration
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

print('Libraries imported successfully.')

Libraries imported successfully.


In [2]:
# Locate repository root and artifacts directories
def find_project_root(start_path: Path) -> Path:
    """Traverse up directory tree to locate repository root containing .env or .git."""
    for parent in [start_path] + list(start_path.parents):
        if (parent / '.env').exists() or (parent / '.git').exists():
            return parent
    return start_path

root_dir = find_project_root(Path.cwd())
load_dotenv(dotenv_path=root_dir / '.env')

# Input artifact from Notebook 1
input_artifact_path = root_dir / 'tasks' / 'task-02-notebooks' / 'artifacts' / '01_read_and_join' / 'joined_orders.parquet'

# Output artifact directory for Notebook 2
output_artifact_dir = root_dir / 'tasks' / 'task-02-notebooks' / 'artifacts' / '02_create_labels'
output_artifact_dir.mkdir(parents=True, exist_ok=True)

print(f'Project root           : {root_dir}')
print(f'Input artifact path    : {input_artifact_path}')
print(f'Output artifact dir    : {output_artifact_dir}')

Project root           : /Users/halakhalifa/Desktop/qafza/olist-mlops
Input artifact path    : /Users/halakhalifa/Desktop/qafza/olist-mlops/tasks/task-02-notebooks/artifacts/01_read_and_join/joined_orders.parquet
Output artifact dir    : /Users/halakhalifa/Desktop/qafza/olist-mlops/tasks/task-02-notebooks/artifacts/02_create_labels


## Step 1: Load Input Artifact from Notebook 1

We load the master table produced in Notebook 1 (`joined_orders.parquet`).

In [3]:
assert input_artifact_path.exists(), f'Input artifact not found at {input_artifact_path}! Run Notebook 1 first.'
df_orders = pd.read_parquet(input_artifact_path)

print(f'Loaded master table: {df_orders.shape[0]:,} rows x {df_orders.shape[1]} columns')
display(df_orders[['order_id', 'order_status', 'order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date']].head())

Loaded master table: 99,441 rows x 33 columns


,order_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26


## Step 2: Population Filtering & Target Eligibility

To formulate a reliable supervised learning target for delivery punctuality, we must only train on orders where the delivery outcome is definitively known and recorded.

We analyze `order_status` and filter out non-delivered or incomplete records.

In [4]:
print('Order Status Distribution across all orders:')
print(df_orders['order_status'].value_counts(dropna=False))

# Check null delivery dates across statuses
null_delivery_by_status = df_orders.groupby('order_status')['order_delivered_customer_date'].apply(lambda s: s.isna().sum())
print('\nMissing customer delivery dates by status:')
print(null_delivery_by_status)

# Filter population: delivered orders with non-null actual and estimated delivery dates
delivered_mask = (df_orders['order_status'] == 'delivered') & (
    df_orders['order_delivered_customer_date'].notna()
) & (
    df_orders['order_estimated_delivery_date'].notna()
)

df_labeled = df_orders[delivered_mask].copy()
dropped_count = len(df_orders) - len(df_labeled)

print(f'\nTotal original orders : {len(df_orders):,}')
print(f'Eligible delivered orders: {len(df_labeled):,} ({len(df_labeled)/len(df_orders)*100:.2f}%)')
print(f'Dropped non-delivered / unfulfilled orders: {dropped_count:,} ({dropped_count/len(df_orders)*100:.2f}%)')

Order Status Distribution across all orders:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Missing customer delivery dates by status:
order_status
approved          2
canceled        619
created           5
delivered         8
invoiced        314
processing      301
shipped        1107
unavailable     609
Name: order_delivered_customer_date, dtype: int64



Total original orders : 99,441
Eligible delivered orders: 96,470 (97.01%)
Dropped non-delivered / unfulfilled orders: 2,971 (2.99%)


## Step 3: Target Construction (`is_late`) & Delivery Timing Metrics

We define the binary target variable **`is_late`**:
- **`is_late = 1` (Late)**: `order_delivered_customer_date > order_estimated_delivery_date`
- **`is_late = 0` (On-Time / Early)**: `order_delivered_customer_date <= order_estimated_delivery_date`

We also derive timing features (in days) to understand delivery dynamics:
- `actual_delivery_days`: Days from purchase to actual delivery (`order_delivered_customer_date - order_purchase_timestamp`).
- `estimated_delivery_days`: Days from purchase to estimated delivery (`order_estimated_delivery_date - order_purchase_timestamp`).
- `delivery_delay_days`: Days past estimate (`order_delivered_customer_date - order_estimated_delivery_date`). Positive indicates late, negative indicates early.

In [5]:
# 1. Create binary target
df_labeled['is_late'] = (
    df_labeled['order_delivered_customer_date'] > df_labeled['order_estimated_delivery_date']
).astype(int)

# 2. Derive delivery duration and delay metrics (in days)
df_labeled['actual_delivery_days'] = (
    df_labeled['order_delivered_customer_date'] - df_labeled['order_purchase_timestamp']
).dt.total_seconds() / 86400.0

df_labeled['estimated_delivery_days'] = (
    df_labeled['order_estimated_delivery_date'] - df_labeled['order_purchase_timestamp']
).dt.total_seconds() / 86400.0

df_labeled['delivery_delay_days'] = (
    df_labeled['order_delivered_customer_date'] - df_labeled['order_estimated_delivery_date']
).dt.total_seconds() / 86400.0

cols_to_inspect = [
    'order_id', 
    'order_purchase_timestamp', 
    'order_delivered_customer_date', 
    'order_estimated_delivery_date', 
    'actual_delivery_days',
    'estimated_delivery_days',
    'delivery_delay_days',
    'is_late'
]

print('=== Verification Sample: 3 Real ON-TIME Orders (is_late == 0) ===')
display(df_labeled[df_labeled['is_late'] == 0][cols_to_inspect].head(3))

print('\n=== Verification Sample: 3 Real LATE Orders (is_late == 1) ===')
display(df_labeled[df_labeled['is_late'] == 1][cols_to_inspect].head(3))


=== Verification Sample: 3 Real ON-TIME Orders (is_late == 0) ===


,order_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,actual_delivery_days,estimated_delivery_days,delivery_delay_days,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,8.436574,15.544063,-7.107488,0
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,13.782037,19.137766,-5.355729,0
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,9.394213,26.639711,-17.245498,0



=== Verification Sample: 3 Real LATE Orders (is_late == 1) ===


,order_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,actual_delivery_days,estimated_delivery_days,delivery_delay_days,is_late
20,203096f03d82e0dffbc41ebc2e2bcfb7,2017-09-18 14:31:30,2017-10-09 22:23:46,2017-09-28,21.327963,9.394792,11.933171,1
25,fbf9ac61453ac646ce8ad9783d7d0af6,2018-02-20 23:46:53,2018-03-21 22:03:54,2018-03-12,28.928484,19.009109,9.919375,1
35,8563039e855156e48fccee4d611a3196,2018-02-17 15:59:46,2018-03-20 00:59:25,2018-03-20,30.374757,30.333495,0.041262,1


## Step 4: Class Imbalance & Delivery Timing Analysis

We analyze the distribution of the target variable and summarize the delivery timing characteristics.

In [6]:
# 1. Create binary target
df_labeled['is_late'] = (
    df_labeled['order_delivered_customer_date'] > df_labeled['order_estimated_delivery_date']
).astype(int)

# 2. Derive delivery duration and delay metrics (in days)
df_labeled['actual_delivery_days'] = (
    df_labeled['order_delivered_customer_date'] - df_labeled['order_purchase_timestamp']
).dt.total_seconds() / 86400.0

df_labeled['estimated_delivery_days'] = (
    df_labeled['order_estimated_delivery_date'] - df_labeled['order_purchase_timestamp']
).dt.total_seconds() / 86400.0

df_labeled['delivery_delay_days'] = (
    df_labeled['order_delivered_customer_date'] - df_labeled['order_estimated_delivery_date']
).dt.total_seconds() / 86400.0

cols_to_inspect = [
    'order_id', 
    'order_purchase_timestamp', 
    'order_delivered_customer_date', 
    'order_estimated_delivery_date', 
    'actual_delivery_days',
    'estimated_delivery_days',
    'delivery_delay_days',
    'is_late'
]

print('=== Verification Sample: 3 Real ON-TIME Orders (is_late == 0) ===')
display(df_labeled[df_labeled['is_late'] == 0][cols_to_inspect].head(3))

print('\n=== Verification Sample: 3 Real LATE Orders (is_late == 1) ===')
display(df_labeled[df_labeled['is_late'] == 1][cols_to_inspect].head(3))


=== Verification Sample: 3 Real ON-TIME Orders (is_late == 0) ===


,order_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,actual_delivery_days,estimated_delivery_days,delivery_delay_days,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,8.436574,15.544063,-7.107488,0
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,13.782037,19.137766,-5.355729,0
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,9.394213,26.639711,-17.245498,0



=== Verification Sample: 3 Real LATE Orders (is_late == 1) ===


,order_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,actual_delivery_days,estimated_delivery_days,delivery_delay_days,is_late
20,203096f03d82e0dffbc41ebc2e2bcfb7,2017-09-18 14:31:30,2017-10-09 22:23:46,2017-09-28,21.327963,9.394792,11.933171,1
25,fbf9ac61453ac646ce8ad9783d7d0af6,2018-02-20 23:46:53,2018-03-21 22:03:54,2018-03-12,28.928484,19.009109,9.919375,1
35,8563039e855156e48fccee4d611a3196,2018-02-17 15:59:46,2018-03-20 00:59:25,2018-03-20,30.374757,30.333495,0.041262,1


### Step 4.2: Late Delivery Rates Across Dimensions

Let's explore how late delivery rates vary across geography (customer state) and time (order month).

In [7]:
# Late rate by top 10 Customer States
state_agg = df_labeled.groupby('customer_state').agg(
    total_orders=('order_id', 'count'),
    late_orders=('is_late', 'sum'),
    late_rate=('is_late', 'mean')
).sort_values('total_orders', ascending=False)
state_agg['late_rate_pct'] = state_agg['late_rate'] * 100

print('Late Delivery Rate by Top Customer States:')
display(state_agg.head(10))

# Late rate over time (by order month)
df_labeled['order_year_month'] = df_labeled['order_purchase_timestamp'].dt.to_period('M').astype(str)
monthly_agg = df_labeled.groupby('order_year_month').agg(
    total_orders=('order_id', 'count'),
    late_orders=('is_late', 'sum'),
    late_rate=('is_late', 'mean')
)
monthly_agg['late_rate_pct'] = monthly_agg['late_rate'] * 100

print('\nMonthly Late Delivery Trend (Sample):')
display(monthly_agg.tail(10))

# Drop helper column before saving
df_labeled = df_labeled.drop(columns=['order_year_month'])

Late Delivery Rate by Top Customer States:


,total_orders,late_orders,late_rate,late_rate_pct
customer_state,,,,
SP,40494,2387,0.058947,5.894700
RJ,12350,1664,0.134737,13.473684
MG,11354,637,0.056104,5.610358
RS,5344,382,0.071482,7.148204
PR,4923,246,0.049970,4.996953
SC,3546,346,0.097575,9.757473
BA,3256,457,0.140356,14.035627
DF,2080,147,0.070673,7.067308
ES,1995,244,0.122306,12.230576



Monthly Late Delivery Trend (Sample):


,total_orders,late_orders,late_rate,late_rate_pct
order_year_month,,,,
2017-11,7288,1043,0.143112,14.311196
2017-12,5513,462,0.083802,8.380192
2018-01,7069,464,0.065639,6.563870
2018-02,6555,1048,0.159878,15.987796
2018-03,7003,1496,0.213623,21.362273
2018-04,6798,361,0.053104,5.310385
2018-05,6749,556,0.082383,8.238258
2018-06,6096,83,0.013615,1.361549
2018-07,6156,276,0.044834,4.483431


## Step 5: Data Integrity & Sanity Checks

We verify:
1. Zero missing values in `is_late`.
2. `is_late` values are strictly in `{0, 1}`.
3. `order_id` is strictly unique.
4. Population size is consistent.

In [8]:
# 1. Create binary target
df_labeled['is_late'] = (
    df_labeled['order_delivered_customer_date'] > df_labeled['order_estimated_delivery_date']
).astype(int)

# 2. Derive delivery duration and delay metrics (in days)
df_labeled['actual_delivery_days'] = (
    df_labeled['order_delivered_customer_date'] - df_labeled['order_purchase_timestamp']
).dt.total_seconds() / 86400.0

df_labeled['estimated_delivery_days'] = (
    df_labeled['order_estimated_delivery_date'] - df_labeled['order_purchase_timestamp']
).dt.total_seconds() / 86400.0

df_labeled['delivery_delay_days'] = (
    df_labeled['order_delivered_customer_date'] - df_labeled['order_estimated_delivery_date']
).dt.total_seconds() / 86400.0

cols_to_inspect = [
    'order_id', 
    'order_purchase_timestamp', 
    'order_delivered_customer_date', 
    'order_estimated_delivery_date', 
    'actual_delivery_days',
    'estimated_delivery_days',
    'delivery_delay_days',
    'is_late'
]

print('=== Verification Sample: 3 Real ON-TIME Orders (is_late == 0) ===')
display(df_labeled[df_labeled['is_late'] == 0][cols_to_inspect].head(3))

print('\n=== Verification Sample: 3 Real LATE Orders (is_late == 1) ===')
display(df_labeled[df_labeled['is_late'] == 1][cols_to_inspect].head(3))


=== Verification Sample: 3 Real ON-TIME Orders (is_late == 0) ===


,order_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,actual_delivery_days,estimated_delivery_days,delivery_delay_days,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,8.436574,15.544063,-7.107488,0
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,13.782037,19.137766,-5.355729,0
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,9.394213,26.639711,-17.245498,0



=== Verification Sample: 3 Real LATE Orders (is_late == 1) ===


,order_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,actual_delivery_days,estimated_delivery_days,delivery_delay_days,is_late
20,203096f03d82e0dffbc41ebc2e2bcfb7,2017-09-18 14:31:30,2017-10-09 22:23:46,2017-09-28,21.327963,9.394792,11.933171,1
25,fbf9ac61453ac646ce8ad9783d7d0af6,2018-02-20 23:46:53,2018-03-21 22:03:54,2018-03-12,28.928484,19.009109,9.919375,1
35,8563039e855156e48fccee4d611a3196,2018-02-17 15:59:46,2018-03-20 00:59:25,2018-03-20,30.374757,30.333495,0.041262,1


## Step 6: Save Artifacts for Notebook 3 (Split)

We save the labeled dataset to `tasks/task-02-notebooks/artifacts/02_create_labels/labeled_orders.parquet`.

In [9]:
labeled_parquet_path = output_artifact_dir / 'labeled_orders.parquet'

# Save Parquet artifact
df_labeled.to_parquet(labeled_parquet_path, index=False)
print(f'✓ Saved labeled artifact: {labeled_parquet_path} ({labeled_parquet_path.stat().st_size / (1024*1024):.2f} MB)')

print('\n=======================================================')
print('✓ Notebook 2 execution completed successfully!')
print('✓ Labeled dataset is ready for Notebook 3 (Train/Val/Test Split).')
print('=======================================================')

✓ Saved labeled artifact: /Users/halakhalifa/Desktop/qafza/olist-mlops/tasks/task-02-notebooks/artifacts/02_create_labels/labeled_orders.parquet (18.49 MB)

✓ Notebook 2 execution completed successfully!
✓ Labeled dataset is ready for Notebook 3 (Train/Val/Test Split).
